# Nextgen AI - Transformer Eğitimi (Google Colab + GPU)

Yerel NumPy eğitimi çok yavaş olduğu için aynı mimariyi **PyTorch (GPU)** ile eğitip ağırlıkları **NumPy `model.json` formatına** dışa aktarıyoruz. Yereldeki `app.py` / `brain.py` hiç değişmeden çalışmaya devam eder.

**Adımlar:**
1. Soldaki **Files** panelinden şu 3 dosyayı `/content` klasörüne yükle:
   - `intents.json`
   - `brain.py`  (kelime/kök üretimi ve veri hazırlama için)
   - `transformer.py`  (NumPy referans modeli, parity doğrulaması için)
2. Tüm hücreleri sırayla çalıştır (Runtime > Run all).
3. Eğitim bitince son hücredeki talimatla `model.json` ve `bot_data.json` dosyalarını indir, yerelde `model/` klasörüne koy.

**Checkpoint/Resume:** Eğitim her 500 adımda (ve her epoch sonunda) Google Drive'a `NextgenAI/checkpoint.pt` yazar. Oturum kesilirse eğitim hücresini yeniden çalıştırman yeterli — en son checkpoint'ten devam eder.

In [ ]:
import sys, os, math, json, time
import numpy as np
sys.path.insert(0, '/content')

# --- Google Drive bagla (checkpoint'ler buraya yazilir) ---
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NextgenAI'
os.makedirs(DRIVE_DIR, exist_ok=True)

# --- Egitim konfigurasyonu ---
EPOCHS      = 500
BATCH_SIZE  = 32
LR_BASE     = 1e-3
LR_MIN      = 0.1          # cosinus cizelgesinin alt siniri
WARMUP      = 200          # ilk N optimizasyon adiminda LR 0 -> LR_BASE
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 5.0
PATIENCE    = 30           # erken durdurma sabri (epoch)
CKPT_EVERY  = 500          # her N adimda Drive'a checkpoint
RESUME      = True         # True: checkpoint varsa devam et
SEED        = 42

import torch
torch.manual_seed(SEED)
np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from brain import ChatBot
bot = ChatBot()
intents_file = '/content/intents.json'
assert os.path.exists(intents_file), 'intents.json /content altinda yok (Files panelinden yukle)'
data = bot.load_intents(intents_file)

# IKI KATMANLI MIMARI: siniflandiriciya yalnizca sohbet intentleri ogretilir
# (deseni >6 olanlar; sayi conversational_data filtresinden gelir). Bilgi
# intentleri (Wikipedia sablonlu "X nedir") anahtar kelime retrieval ile
# cevaplanir. intent_tags sohbet siniflarina iner ama intents/intent_kws 791
# intent icin dolu kalir -> yereldeki brain.py bilgi sorularini retrieval ile
# yanitlar.
data = bot.conversational_data(data)
print(f'Sohbet intentleri: {len(bot.intent_tags)} | Bilgi intentleri (retrieval): {len(bot.knowledge_intents)}')
X, y = bot.prepare_training_data(data)
MAX_SEQ_LEN  = int(bot.max_seq_len)
VOCAB_SIZE   = len(bot.vocabulary)
NUM_INTENTS  = len(bot.intent_tags)

# Sabit tohumla tekrarlanabilir train/val ayrimi (erken durdurma val'de olculur)
n_val = max(1, int(0.1 * len(X)))
perm = np.random.RandomState(42).permutation(len(X))
Xtr, ytr = X[perm[n_val:]], y[perm[n_val:]]
Xval, yval = X[perm[:n_val]], y[perm[:n_val]]
print('X', X.shape, '| train', Xtr.shape, '| val', Xval.shape, '| vocab', VOCAB_SIZE, '| intents', NUM_INTENTS, '| seq', MAX_SEQ_LEN)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class TorchMHA(nn.Module):
    """transformer.py'deki MultiHeadAttention'un birebir ayrisi."""
    def __init__(self, d, heads, attn_dropout=0.05):
        super().__init__()
        self.d, self.heads, self.hd = d, heads, d // heads
        self.attn_dropout = attn_dropout
        def lin(scale):
            l = nn.Linear(d, d)
            with torch.no_grad():
                l.weight.normal_(0, scale)
                l.bias.zero_()
            return l
        s = math.sqrt(2.0 / d)
        self.Wq, self.Wk, self.Wv = lin(s), lin(s), lin(s)
        self.Wo = lin(0.02)
        self.rsqrt = 1.0 / math.sqrt(self.hd)

    def forward(self, x, mask):
        B, L, d = x.shape
        H, hd = self.heads, self.hd
        Q = self.Wq(x); K = self.Wk(x); V = self.Wv(x)
        Qh = Q.view(B, L, H, hd).transpose(1, 2)
        Kh = K.view(B, L, H, hd).transpose(1, 2)
        Vh = V.view(B, L, H, hd).transpose(1, 2)
        scores = (Qh @ Kh.transpose(-1, -2)) * self.rsqrt
        row = mask[:, None, :, None]                     # (B,1,L,1)
        valid = row * mask[:, None, None, :]             # (B,1,L,L)
        logits = scores.masked_fill(valid <= 0, -1e9)
        p = F.softmax(logits, dim=-1) * row              # PAD sorgu satirlari 0
        if self.training and self.attn_dropout > 0:
            p = F.dropout(p, self.attn_dropout)
        out = (p @ Vh).transpose(1, 2).contiguous().view(B, L, d)
        return self.Wo(out)


class TorchBlock(nn.Module):
    """Pre-LN bloğu: LN1 -> MHA -> +residual ; LN2 -> GELU-FFN -> +residual"""
    def __init__(self, d, heads, ff_dim, dropout=0.1, attn_dropout=0.05):
        super().__init__()
        self.ln1 = nn.LayerNorm(d, eps=1e-5)
        self.attn = TorchMHA(d, heads, attn_dropout)
        self.ln2 = nn.LayerNorm(d, eps=1e-5)
        s = math.sqrt(2.0 / d)
        self.W1 = nn.Linear(d, ff_dim)
        self.W2 = nn.Linear(ff_dim, d)
        self.dropout = dropout
        with torch.no_grad():
            self.W1.weight.normal_(0, s); self.W1.bias.zero_()
            self.W2.weight.normal_(0, 0.02); self.W2.bias.zero_()

    def forward(self, x, mask):
        a = x + self.attn(self.ln1(x), mask)
        h = F.gelu(self.W1(self.ln2(a)), approximate='tanh')
        if self.training and self.dropout > 0:
            h = F.dropout(h, self.dropout)
        return a + self.W2(h)


class TorchTransformer(nn.Module):
    """transformer.py'deki TransformerNN'in birebir PyTorch karsiligi."""
    def __init__(self, vocab_size, num_intents, max_seq_len, d=128, blocks=4,
                 heads=4, ff_mult=4, dropout=0.1, attn_dropout=0.05):
        super().__init__()
        self.vocab_size, self.pad_idx = vocab_size, vocab_size
        self.num_intents = num_intents
        self.max_seq_len = max_seq_len
        self.d_model, self.num_blocks, self.num_heads = d, blocks, heads
        self.ff_dim = ff_mult * d
        self.dropout, self.attn_dropout = dropout, attn_dropout
        self.embed = nn.Embedding(vocab_size + 1, d)
        self.register_buffer('pos', self._sinusoidal(max_seq_len))
        self.blocks = nn.ModuleList(
            [TorchBlock(d, heads, self.ff_dim, dropout, attn_dropout)
             for _ in range(blocks)])
        self.Whead = nn.Linear(d, num_intents)
        with torch.no_grad():
            self.embed.weight[:vocab_size].normal_(0, 0.02)
            self.embed.weight[vocab_size].zero_()
            self.Whead.weight.normal_(0, 0.1)
            self.Whead.bias.zero_()

    def _sinusoidal(self, length):
        d = self.d_model
        pe = torch.zeros(length, d)
        pos = torch.arange(length, dtype=torch.float32).unsqueeze(1)
        dim = torch.arange(d // 2, dtype=torch.float32)
        div = torch.pow(torch.tensor(10000.0, dtype=torch.float32),
                        2.0 * dim / d)
        pe[:, 0::2] = torch.sin(pos / div)
        pe[:, 1::2] = torch.cos(pos / div)
        pe *= (1.0 / math.sqrt(max(d, 1)))   # konumsal embedlerin (1/sqrt(d)) olcegi
        return pe

    def forward(self, X):
        B, L = X.shape
        mask = (X != self.pad_idx).float()
        mask_ = mask[:, :, None]
        x = self.embed(X) * mask_
        x = x + self.pos[:L][None] * mask_
        if self.training and self.dropout > 0:
            x = F.dropout(x, self.dropout)
        for blk in self.blocks:
            x = blk(x, mask)
        denom = mask.sum(1, keepdim=True).clamp(min=1.0)
        pooled = (x * mask_).sum(1) / denom
        head_out = self.Whead(pooled)
        return F.softmax(head_out, dim=-1)

    def logits(self, X):
        B, L = X.shape
        mask = (X != self.pad_idx).float()
        mask_ = mask[:, :, None]
        x = self.embed(X) * mask_
        x = x + self.pos[:L][None] * mask_
        if self.training and self.dropout > 0:
            x = F.dropout(x, self.dropout)
        for blk in self.blocks:
            x = blk(x, mask)
        denom = mask.sum(1, keepdim=True).clamp(min=1.0)
        return self.Whead((x * mask_).sum(1) / denom)


In [ ]:
# ------------------- Checkpoint / Resume yardimcilari -------------------
CKPT_PATH = os.path.join(DRIVE_DIR, 'checkpoint.pt')   # en son adim (resume)
BEST_PATH = os.path.join(DRIVE_DIR, 'best.pt')         # en iyi dogruluk
FINAL_PATH = os.path.join(DRIVE_DIR, 'final.pt')       # egitim bittiginde

def save_checkpoint(path, model, optimizer, global_step, epoch, best_acc):
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'global_step': global_step,
        'epoch': epoch,
        'best_acc': best_acc,
        'torch_rng': torch.random.get_rng_state(),
        'np_rng': np.random.get_state(),
    }, path)

def load_checkpoint(path, model, optimizer, device):
    ck = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    torch.random.set_rng_state(ck['torch_rng'].cpu())
    np.random.set_state(tuple(ck['np_rng']))
    return ck['global_step'], ck['epoch'], ck['best_acc']

def build_model():
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = TorchTransformer(VOCAB_SIZE, NUM_INTENTS, MAX_SEQ_LEN,
                         d=128, blocks=4, heads=4, ff_mult=4,
                         dropout=0.1, attn_dropout=0.05).to(device)
    decay, no_decay = [], []
    for n, p in m.named_parameters():
        (no_decay if p.ndim == 1 else decay).append(p)  # bias/LN ayrismaz
    opt = torch.optim.AdamW(
        [{'params': decay, 'weight_decay': WEIGHT_DECAY},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=LR_BASE, betas=(0.9, 0.999), eps=1e-8)
    return m, opt

Xt = torch.from_numpy(Xtr).to(device)
yt = torch.from_numpy(ytr.astype(np.int64)).to(device)
Xv = torch.from_numpy(Xval).to(device)
yv = torch.from_numpy(yval.astype(np.int64)).to(device)
N = Xtr.shape[0]
n_batches = (N + BATCH_SIZE - 1) // BATCH_SIZE
total_steps = EPOCHS * n_batches
def scheduled_lr(step):
    if WARMUP > 0 and step < WARMUP:
        return LR_BASE * (step + 1.0) / WARMUP
    w = max(step - WARMUP, 0)
    total = max(total_steps - WARMUP, 1)
    frac = w / total
    return LR_BASE * (LR_MIN + (1 - LR_MIN) * 0.5 * (1 + math.cos(math.pi * frac)))

In [ ]:
model, optimizer = build_model()
criterion = torch.nn.NLLLoss()

global_step, start_epoch, best_acc = 0, 0, 0.0
if RESUME and os.path.exists(CKPT_PATH):
    global_step, start_epoch, best_acc = load_checkpoint(CKPT_PATH, model, optimizer, device)
    print(f'[RESUME] epoch={start_epoch+1}/{EPOCHS} adim={global_step} best_acc={best_acc:.4f}')
else:
    print('[FRESH] yeni egitim basliyor')
epoch = start_epoch

def evaluate_acc():
    """Dogruluk DOGRULAMA setinde olculur (egitim seti ezberi yaniltir)."""
    model.eval()
    n_correct = 0
    with torch.no_grad():
        for st in range(0, len(Xval), 256):
            xb = Xv[st:st+256]
            pred = model(xb).argmax(1)
            n_correct += int((pred == yv[st:st+256]).sum())
    model.train()
    return n_correct / len(Xval)

t_start = time.time()
save_next = (global_step // CKPT_EVERY + 1) * CKPT_EVERY
wait = 0
model.train()

for epoch in range(start_epoch, EPOCHS):
    perm = np.random.permutation(N)
    ep_loss, nb = 0.0, 0
    t_ep = time.time()
    for st in range(0, N, BATCH_SIZE):
        idx = perm[st:st+BATCH_SIZE]
        xb, yb = Xt[idx], yt[idx]
        for g in optimizer.param_groups:
            g['lr'] = scheduled_lr(global_step)
        optimizer.zero_grad(set_to_none=True)
        probs = model(xb)
        loss = criterion(torch.log(probs.clamp_min(1e-8)), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        global_step += 1
        ep_loss += float(loss.item()); nb += 1
        if global_step >= save_next:
            save_checkpoint(CKPT_PATH, model, optimizer, global_step, epoch + 1, best_acc)
            print(f'  [ckpt] adim {global_step} -> {DRIVE_DIR}')
            save_next += CKPT_EVERY
    acc = evaluate_acc()
    dt = time.time() - t_ep
    eta = (EPOCHS - epoch - 1) * dt
    print(f'E{epoch+1:3d}/{EPOCHS} loss={ep_loss/nb:.4f} acc={acc*100:5.2f}% '
          f'lr={optimizer.param_groups[0]["lr"]:.5f} {dt:.1f}s/epoch ETA={eta/60:.1f}dk')
    save_checkpoint(CKPT_PATH, model, optimizer, global_step, epoch + 1, best_acc)
    if acc > best_acc + 1e-4:
        best_acc = acc; wait = 0
        torch.save({'model': model.state_dict(), 'best_acc': best_acc,
                    'step': global_step, 'epoch': epoch + 1}, BEST_PATH)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f'Erken durdurma: epoch {epoch+1} (en iyi acc {best_acc:.4f})')
            break

if os.path.exists(BEST_PATH):
    ck = torch.load(BEST_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ck['model'])
    best_acc = ck['best_acc']
model.eval()
torch.save({'model': model.state_dict(), 'best_acc': best_acc,
            'step': global_step, 'epoch': epoch + 1}, FINAL_PATH)
print(f'Toplam {time.time()-t_start:.0f}s | en iyi acc {best_acc:.4f}')
print(f'Model Drive konumu: {FINAL_PATH} ({os.path.getsize(FINAL_PATH)/1e6:.1f} MB)')

In [ ]:
# ------------------- NumPy model.json + bot_data.json export -------------------
def export_numpy_model(model, bot, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    st = model.state_dict()
    params = {}
    params['embed'] = st['embed.weight'].cpu().numpy()
    for i in range(model.num_blocks):
        for nm, src in [('Wq','attn.Wq.weight'),('Wk','attn.Wk.weight'),
                        ('Wv','attn.Wv.weight'),('Wo','attn.Wo.weight')]:
            params[f'b{i}_{nm}'] = st[f'blocks.{i}.{src}'].cpu().numpy().T
        for nm, src in [('bq','attn.Wq.bias'),('bk','attn.Wk.bias'),
                        ('bv','attn.Wv.bias'),('bo','attn.Wo.bias')]:
            params[f'b{i}_{nm}'] = st[f'blocks.{i}.{src}'].cpu().numpy()[None, :]
        for nm, src in [('ln1_g','ln1.weight'),('ln1_b','ln1.bias'),
                        ('ln2_g','ln2.weight'),('ln2_b','ln2.bias')]:
            params[f'b{i}_{nm}'] = st[f'blocks.{i}.{src}'].cpu().numpy()[None, :]
        params[f'b{i}_W1'] = st[f'blocks.{i}.W1.weight'].cpu().numpy().T
        params[f'b{i}_b1'] = st[f'blocks.{i}.W1.bias'].cpu().numpy()[None, :]
        params[f'b{i}_W2'] = st[f'blocks.{i}.W2.weight'].cpu().numpy().T
        params[f'b{i}_b2'] = st[f'blocks.{i}.W2.bias'].cpu().numpy()[None, :]
    params['Whead'] = st['Whead.weight'].cpu().numpy().T
    params['bhead'] = st['Whead.bias'].cpu().numpy()[None, :]
    data = {'arch': 'transformer', 'vocab_size': model.vocab_size,
            'num_intents': model.num_intents, 'max_seq_len': model.max_seq_len,
            'd_model': model.d_model, 'num_blocks': model.num_blocks,
            'num_heads': model.num_heads, 'ff_dim': model.ff_dim,
            'dropout': model.dropout, 'attn_dropout': model.attn_dropout,
            'weight_decay': float(WEIGHT_DECAY), 'max_grad_norm': float(GRAD_CLIP),
            'weights_file': 'model_weights.npz'}
    with open(os.path.join(out_dir, 'model.json'), 'w', encoding='utf-8') as f:
        json.dump(data, f)
    np.savez(os.path.join(out_dir, 'model_weights.npz'),
             **{k: v.astype(np.float32) for k, v in params.items()})
    bot_data = {'vocabulary': bot.vocabulary, 'intent_tags': bot.intent_tags,
                'intents': bot.intents,
                'intent_kws': {t: sorted(list(k)) for t, k in bot.intent_kws.items()}}
    with open(os.path.join(out_dir, 'bot_data.json'), 'w', encoding='utf-8') as f:
        json.dump(bot_data, f, ensure_ascii=False, indent=2)
    print('Export tamam:', os.path.abspath(out_dir))

export_numpy_model(model, bot, os.path.join(DRIVE_DIR, 'export'))
export_numpy_model(model, bot, '/content/model')

In [ ]:
# ------------------- Parity dogrulama: torch vs numpy -------------------
from transformer import TransformerNN   # NumPy referans modeli (yuklenen dosya)
mx = TransformerNN(vocab_size=VOCAB_SIZE, num_intents=NUM_INTENTS,
                   max_seq_len=MAX_SEQ_LEN, d_model=128, num_blocks=4,
                   num_heads=4, ff_mult=4, dropout=0.1, attn_dropout=0.05)
mx.load('/content/model/model.json')

with torch.no_grad():
    tprobs = model(Xt[:64]).cpu().numpy()
nprobs = mx.predict_proba(Xtr[:64])
diff = float(np.abs(tprobs - nprobs).max())
print('max |torch - numpy| on 64 ornek:', diff)
assert diff < 2e-5, 'PARITY FAIL'
print('PARITY OK (torch ile numpy birebir tutarli)')

def pad_seq(idx):
    idx = list(idx)[:MAX_SEQ_LEN]
    return np.array(idx + [VOCAB_SIZE] * (MAX_SEQ_LEN - len(idx)), dtype=np.int64)

for q in ['merhaba', 'fizik nedir', 'ankara hava durumu', 'pizza nasil yapilir',
          'yapay zeka hakkinda bilgi ver']:
    p = mx.predict_proba(pad_seq(bot.text_to_indices(q))[None, :])[0]
    best = bot.intent_tags[int(np.argmax(p))]
    print(f'{q} -> {best} ({p.max():.2%})')

## İndirme ve yerel kurulum

**Eğitim bittiğinde / checkpoint'ten devam etmek istiyorsan:**

- **Eğitim kesildiyse:** Oturumu kapat/aç, dosyalar hâlâ Drive'daysa sadece **setup + data-pre + model + eğitim** hücrelerini yeniden çalıştır. `RESUME=True` olduğu için `NextgenAI/checkpoint.pt` otomatik yüklenir ve kaldığı yerden devam eder.

**Yerelden kullanmak için:**

1. Logistic:
   - `NextgenAI/export/model.json`, `NextgenAI/export/model_weights.npz` ve `NextgenAI/export/bot_data.json`
   - (veya sol Files panelden `/content/model/model.json` + `model_weights.npz` + `bot_data.json`'i indir)
2. Bu üç dosyayı yerel `model/` klasöründekilerin **üstüne kopyala** (eski bozuk `model.json` ezilir; `_backup_feedforward/` yedeği dokunulmaz).
3. Test: `python excel_predict.py` veya `python app.py`.

**Not:** Aynı GPU oturumunda eğitimi tekrarlarsan checkpoint'ten devam eder; tamamen sıfırdan istersen `RESUME=False` yap veya Drive'daki `checkpoint.pt`'yi sil.